# Imports and Setup

In [ ]:
%load_ext autoreload

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s') # NOTSET, DEBUG, INFO, WARN, ERROR, CRITICAL

import os
import numpy as np

import jax
import jax.numpy as jnp
from jax import vmap, jit, grad, jacfwd

# Optional: higher numeric accuracy (do this before heavy JAX use)
# try:
#     from jax import config as jax_config
#     jax_config.update("jax_enable_x64", True)
# except Exception:
#     pass

import skewerjax as sjx

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
plt.style.use('tableau-colorblind10')
font, rcnew = sjx.config.setup_matplotlib.matplotlib_default_config()
plt.rc('font', **font)
plt.rcParams.update(rcnew)
%matplotlib widget

In [ ]:
from skewerjax.skewers.skewers import SkewerBundle

from skewerjax.fields.fields import ScalarField, VectorField, Fields
import skewerjax.fields.toy_model_samplers as toy_samplers
import skewerjax.utils.aux as aux

from skewerjax.optical_depth import line_profile_velocity_units

from skewerjax.constants.physical_constants import phys
from skewerjax.optical_depth.voigt import Lorentzian

# Validation against analytical result with $T=0,\, v_{LoS}$=0, a=0.9995

### Define the fields to sample

In [ ]:
L_box = 1.0 # in Mpc

T0 = jnp.finfo(jnp.dtype('float32')).tiny
v0 = 0.0

fields = Fields({
    "HI": {
        "temperature": ScalarField(toy_samplers.temperature_sampler(T0_min=T0, T0_max=T0, L_box=L_box)), # UNITS: [K]
        "velocity": VectorField(toy_samplers.velocity_sampler(v0_min=v0, v0_max=v0, L_box=L_box)),       # UNITS: [km/s]
        "turbulence": ScalarField(toy_samplers.turbulence_sampler(b_turb=0.0)),                          # UNITS: [km/s]
        "collisional": ScalarField(toy_samplers.collisional_sampler(A_coll=0.0))                         # UNITS: [GHz]
    },
    "SiIII": {
        "temperature": ScalarField(toy_samplers.temperature_sampler(T0_min=T0, T0_max=T0, L_box=L_box)), # UNITS: [K]
        "velocity": VectorField(toy_samplers.velocity_sampler(v0_min=v0, v0_max=v0, L_box=L_box)),       # UNITS: [km/s]
        "turbulence": ScalarField(toy_samplers.turbulence_sampler(b_turb=0.0)),                          # UNITS: [km/s]
        "collisional": ScalarField(toy_samplers.collisional_sampler(A_coll=0.0))                         # UNITS: [GHz]
    }
})

### Define velocity range and lines to study

In [ ]:
velocities_kms = jnp.linspace(-3000.0, 3000.0, 5*10**4)
line_bundle = sjx.constants.lines.LineBundle(["HI-LyA", "HI-LyB", "SiIII-1206"])

### Define points to evaluate

In [ ]:
quadri_vectors = jnp.array([0.995, L_box/2, L_box/2, L_box/2])[None, :] # single points at the center of the box, with scale_factor (first entry) = 1.0
LoS_directions = jnp.array([[0.0, 0.0, 1.0]])                         # line of sight direction is along the z-axis

### Compute line profile

In [ ]:
a_dagger = jnp.ones_like(quadri_vectors[0, 0])

In [ ]:
phi_ns = line_profile_velocity_units.compute_line_profiles_velocity_units(
    quadri_vectors, LoS_directions, velocities_kms, a_dagger, line_bundle, fields, voigt_implementation_mode="ExoJAX",
)

### Validation plot

In [ ]:
scale_factor_obs = np.ones(line_bundle.n_lines)

wavelengths_per_line_A = line_profile_velocity_units.velocities_to_line_centered_wavelengths(
    velocities_kms, line_bundle, a_dagger, a_obs=scale_factor_obs,
)
lambdas_A_prime = line_profile_velocity_units.compute_lambda_prime_wavelength_units(
    wavelengths_A=wavelengths_per_line_A, scale_factor_obs=scale_factor_obs, scale_factor=quadri_vectors[:, 0],
    LoS_velocity=jnp.zeros((line_bundle.n_lines, quadri_vectors.shape[0])),
)

gamma_GHz = jnp.asarray(line_bundle.A21_GHz) / (4.0 * jnp.pi)
lambda0_A = jnp.asarray(line_bundle.lambda0_A)

scale_factor = np.ones(line_bundle.n_lines) * quadri_vectors[:, 0]

lambda0_A_expanded = lambda0_A[:, None, None]

nu_GHz = ((1.0 / lambdas_A_prime) - (1.0 / lambda0_A_expanded)) * phys.c * 10.0

gamma_GHz_expanded = gamma_GHz[:, None, None]

phi_analytic = Lorentzian(nu_GHz, gamma_GHz_expanded)

In [ ]:
colors = np.array(plt.cm.get_cmap("tab10").colors[:line_bundle.n_lines])
v_np, phi_np = np.asarray(velocities_kms), np.asarray(phi_ns)
N_lines, N_v = line_bundle.n_lines, len(velocities_kms)

phi_plot = phi_np if phi_np.ndim == 2 else (phi_np[:, 0, :] if phi_np.ndim == 3 and phi_np.shape[1] == 1 and phi_np.shape[-1] == N_v else phi_np.reshape(N_lines, -1, N_v)[:, 0, :])

try:
    phi_analytic_plot = np.asarray(phi_analytic)
    phi_analytic_plot = phi_analytic_plot if phi_analytic_plot.ndim == 2 else (phi_analytic_plot[:, 0, :] if phi_analytic_plot.ndim == 3 and phi_analytic_plot.shape[1] == 1 and phi_analytic_plot.shape[-1] == N_v else phi_analytic_plot.reshape(N_lines, -1, N_v)[:, 0, :])
except NameError:
    phi_analytic_plot = None

try:
    a_obs = np.asarray(scale_factor_obs)
    a_obs = np.full(N_lines, a_obs) if a_obs.ndim == 0 else a_obs.reshape(-1)[:N_lines]
except NameError:
    a_obs = np.ones(N_lines)

a_dagger_np, a_los = np.asarray(a_dagger), np.asarray(quadri_vectors[0, 0])
a_dagger_lines = np.full(N_lines, a_dagger_np) if np.asarray(a_dagger_np).ndim == 0 else np.asarray(a_dagger_np).reshape(-1)[:N_lines]
c_kms = phys.c * 1e-3

v_center_kms = c_kms * np.log(a_dagger_np / a_los)
lambda0_A = np.asarray(line_bundle.lambda0_A)
lambdas_per_line_A = lambda0_A[:, None] * (a_obs[:, None] / a_dagger_lines[:, None]) * np.exp(v_np[None, :] / c_kms)
lambda_center_A = lambda0_A * (a_obs / a_dagger_lines) * np.exp(v_center_kms / c_kms)

res_band, res_linthresh = 1e-6, 1e-6
res_ticks = [-1e-2, -1e-5, 0.0, 1e-5, 1e-2]
fmt = sjx.utils.plotting_tools.ten_power_formatter

# ------------------------------------------------------------
# Figure 1: velocity units, one column per line + residuals
# ------------------------------------------------------------
fig_v, axes_v = plt.subplots(2, N_lines, figsize=(5.0 * N_lines, 6.8), sharex="col", sharey="row", gridspec_kw=dict(height_ratios=[2.4, 1.0]))
axes_v = axes_v.reshape(2, 1) if N_lines == 1 else axes_v

for i, line in enumerate(line_bundle.lines):
    ax, rx = axes_v[0, i], axes_v[1, i]
    ax.plot(v_np, phi_plot[i], color=colors[i], linestyle="-", linewidth=2.0)
    if phi_analytic_plot is not None: ax.plot(v_np, phi_analytic_plot[i], color=colors[i], linestyle=":", linewidth=3.0, alpha=0.7)
    ax.scatter([v_center_kms], [4.0 / line.A21_GHz], marker="x", color=colors[i], s=100)
    ax.axvline(0.0, color=colors[i], linestyle="-.", linewidth=1.5, alpha=0.9)
    ax.axvline(v_center_kms, color=colors[i], linestyle="--", linewidth=1.5, alpha=0.9)
    ax.set_title(rf"{line.name}, $\lambda_0={line.lambda0_A:.3f}\,\mathrm{{\AA}}$", fontsize=16)
    ax.set_yscale("log")
    ax.set_ylabel(r"$\phi\;[\mathrm{ns}]$")

    rx.axvline(0.0, color=colors[i], linestyle="-.", linewidth=1.2, alpha=0.9)
    rx.axvline(v_center_kms, color=colors[i], linestyle="--", linewidth=1.2, alpha=0.9)
    rx.axhline(0.0, color="k", linewidth=1.0, alpha=0.5)
    rx.fill_between(v_np, -res_band, res_band, color="grey", alpha=0.2)
    rx.hlines([res_band, -res_band], xmin=v_np[0], xmax=v_np[-1], color="k", linewidth=1.0, linestyle="--", alpha=0.5)
    if phi_analytic_plot is not None: rx.plot(v_np, phi_plot[i] / phi_analytic_plot[i] - 1.0, color=colors[i], linewidth=1.0)
    rx.set_xlabel(r"$v\;[\mathrm{km\,s^{-1}}]$")
    rx.set_ylabel("residual")
    rx.set_yscale("symlog", linthresh=res_linthresh)
    rx.set_yticks(res_ticks)
    rx.yaxis.set_major_formatter(mpl.ticker.FuncFormatter(fmt))
    rx.yaxis.set_minor_formatter(mpl.ticker.NullFormatter())

handles_v = [
    Line2D([0], [0], color="k", linestyle="-", linewidth=2.0, label="SkewerJAX"),
    Line2D([0], [0], color="k", linestyle=":", linewidth=5.0, label="Analytical"),
    Line2D([0], [0], color="k", marker="x", linestyle="None", markersize=9, markeredgewidth=2, label=r"$4/A_{21}$"),
    Line2D([0], [0], color="k", linestyle="--", linewidth=1.5, label="absorber center"),
    Line2D([0], [0], color="k", linestyle="-.", linewidth=1.5, label="rest-frame center"),
]
if phi_analytic_plot is None: handles_v = [h for h in handles_v if h.get_label() != "Analytical"]
axes_v[0, 0].legend(handles=handles_v, fontsize=14, loc="best", frameon=True)
fig_v.tight_layout()
fig_v.subplots_adjust(hspace=0.05, wspace=0.2)
plt.show()

# ------------------------------------------------------------
# Figure 2: wavelength units, all lines together + residuals
# ------------------------------------------------------------
fig_l, (ax_l, rx_l) = plt.subplots(2, 1, figsize=(14, 8), sharex=True, gridspec_kw=dict(height_ratios=[2.4, 1.0]))

for i, line in enumerate(line_bundle.lines):
    ax_l.plot(lambdas_per_line_A[i], phi_plot[i], color=colors[i], linestyle="-", linewidth=2.0)
    if phi_analytic_plot is not None: ax_l.plot(lambdas_per_line_A[i], phi_analytic_plot[i], color=colors[i], linestyle=":", linewidth=3.0, alpha=0.7)
    ax_l.scatter([lambda_center_A[i]], [4.0 / line.A21_GHz], marker="x", color=colors[i], s=100)
    ax_l.axvline(line.lambda0_A, color=colors[i], linestyle="-.", linewidth=1.3, alpha=0.8)
    ax_l.axvline(lambda_center_A[i], color=colors[i], linestyle="--", linewidth=1.3, alpha=0.8)

    rx_l.axvline(line.lambda0_A, color=colors[i], linestyle="-.", linewidth=1.1, alpha=0.8)
    rx_l.axvline(lambda_center_A[i], color=colors[i], linestyle="--", linewidth=1.1, alpha=0.8)
    if phi_analytic_plot is not None: rx_l.plot(lambdas_per_line_A[i], phi_plot[i] / phi_analytic_plot[i] - 1.0, color=colors[i], linewidth=1.0, alpha=0.7)

x_min, x_max = float(np.min(lambdas_per_line_A)), float(np.max(lambdas_per_line_A))
ax_l.set_xlim(x_min, x_max)
ax_l.set_yscale("log")
ax_l.set_ylabel(r"$\phi\;[\mathrm{ns}]$")

rx_l.axhline(0.0, color="k", linewidth=1.0, alpha=0.5)
rx_l.fill_between([x_min, x_max], -res_band, res_band, color="grey", alpha=0.2)
rx_l.hlines([res_band, -res_band], xmin=x_min, xmax=x_max, color="k", linewidth=1.0, linestyle="--", alpha=0.5)
rx_l.set_xlabel(r"$\lambda_{\rm obs}\;[\mathrm{\AA}]$")
rx_l.set_ylabel("residual")
rx_l.set_yscale("symlog", linthresh=res_linthresh)
rx_l.set_yticks(res_ticks)
rx_l.yaxis.set_major_formatter(mpl.ticker.FuncFormatter(fmt))
rx_l.yaxis.set_minor_formatter(mpl.ticker.NullFormatter())

handles_l = [
    Line2D([0], [0], color="k", linestyle="-", linewidth=2.0, label="SkewerJAX"),
    Line2D([0], [0], color="k", linestyle=":", linewidth=5.0, label="Analytical"),
    Line2D([0], [0], color="k", marker="x", linestyle="None", markersize=9, markeredgewidth=2, label=r"$4/A_{21}$"),
    Line2D([0], [0], color="k", linestyle="-.", linewidth=1.5, label=r"rest wavelength $\lambda_0$"),
    Line2D([0], [0], color="k", linestyle="--", linewidth=1.5, label="absorber center"),
]
if phi_analytic_plot is None: handles_l = [h for h in handles_l if h.get_label() != "Analytical"]
ax_l.add_artist(ax_l.legend(handles=handles_l, loc="lower left", fontsize=14, frameon=True, ncols=5))

line_handles = [Line2D([0], [0], color=colors[i], linestyle="-", linewidth=3.0, label=rf"{line.name}, $\lambda_0={line.lambda0_A:.3f}\,\mathrm{{\AA}}$") for i, line in enumerate(line_bundle.lines)]
ax_l.legend(handles=line_handles, loc="upper right", fontsize=14, frameon=True, ncols=len(line_handles))

fig_l.tight_layout()
fig_l.subplots_adjust(hspace=0.05)
plt.show()

# Check against analytical results varying scale factors

### Define events with different scale factor combinations $(a(x), a_0)$

In [ ]:
quadri_vectors = jnp.array([             # quadri-vectors defined at the center of the volume considering different scale factors
    [0.999, L_box/2, L_box/2, L_box/2],
    [0.5,   L_box/2, L_box/2, L_box/2],
    [0.5,   L_box/2, L_box/2, L_box/2],
    [1.0,   L_box/2, L_box/2, L_box/2],
    [1.0,   L_box/2, L_box/2, L_box/2]
])

a_dagger = jnp.array([           # scale factors wrt the observer
    1.0, 0.5, 0.501, 1.003, 0.999
]) 

LoS_directions = jnp.zeros((quadri_vectors.shape[0], 3))
LoS_directions = LoS_directions.at[..., 0].set(1.0)      # line-of-sight directions corresponding to N_sk skewers pointing towards the z-direction, shape (N_sk, N_ppsk, 3)

In [ ]:
phi_ns = line_profile_velocity_units.compute_line_profiles_velocity_units(
    quadri_vectors, LoS_directions, velocities_kms, a_dagger, line_bundle, fields, voigt_implementation_mode="ExoJAX",
)

In [ ]:
N_lines = line_bundle.n_lines
N_points = quadri_vectors.shape[0]
N_v = velocities_kms.size

lambda0_A = jnp.asarray(line_bundle.lambda0_A)      # (N_lines,)
gamma_GHz = jnp.asarray(line_bundle.A21_GHz) / (4.0 * jnp.pi)

scale_factor = quadri_vectors[:, 0]                 # (N_points,)
LoS_velocity_zero = jnp.zeros((N_lines, N_points))  # (N_lines, N_points)

lambdas_A_prime = line_profile_velocity_units.compute_lambda_prime_velocity_units(
    velocities_kms=velocities_kms,
    lambda0_A=lambda0_A,
    a_dagger=a_dagger,              # (N_points,)
    scale_factor=scale_factor,      # (N_points,)
    LoS_velocity=LoS_velocity_zero, # (N_lines, N_points)
)

# Shape: (N_lines, N_points, N_v)
print("lambdas_A_prime.shape =", lambdas_A_prime.shape)

nu_GHz = (
    (1.0 / lambdas_A_prime)
    - (1.0 / lambda0_A[:, None, None])
) * phys.c * 10.0

gamma_GHz_expanded = gamma_GHz[:, None, None]

phi_analytic = Lorentzian(nu_GHz, gamma_GHz_expanded)

print("phi_analytic.shape =", phi_analytic.shape)
# Expected: (N_lines, N_points, N_v)

In [ ]:
# Convert JAX arrays to NumPy for plotting
velocities_np = np.asarray(velocities_kms)
phi_np = np.asarray(phi_ns)
phi_analytic_np = np.asarray(phi_analytic)

N_lines, N_points = line_bundle.n_lines, quadri_vectors.shape[0]
colors = np.array(plt.cm.get_cmap("tab10").colors[:N_points])
scale_factor = np.asarray(quadri_vectors[:, 0])
a_dagger_np = np.asarray(a_dagger)
a_dagger_per_point = np.full((N_points,), float(a_dagger_np)) if np.ndim(a_dagger_np) == 0 else a_dagger_np
c_kms = phys.c * 1e-3

mask = np.ones_like(velocities_np, dtype=bool)
v_win = velocities_np[mask]

fig, axes = plt.subplots(nrows=2, ncols=N_lines, figsize=(14, 8), sharex=True, sharey="row", gridspec_kw=dict(height_ratios=[2, 1]))
if N_lines == 1: axes = axes.reshape(2, 1)

for i_line, (ax_main, ax_res, line) in enumerate(zip(axes[0], axes[1], line_bundle.lines)):
    ax_main.axvline(0.0, color="k", linestyle="-", linewidth=2.5, alpha=0.7); ax_res.axvline(0.0, color="k", linestyle="-", linewidth=1.5, alpha=0.5)

    for i_point in range(N_points):
        v_peak = c_kms * np.log(a_dagger_per_point[i_point] / scale_factor[i_point])
        peak_height = 4.0 / line_bundle.lines[i_line].A21_GHz

        ax_main.scatter([v_peak], [peak_height], marker="x", color=colors[i_point], s=100, alpha=1.0)
        ax_main.axvline(v_peak, color=colors[i_point], linestyle="-.", linewidth=1.5); ax_res.axvline(v_peak, color=colors[i_point], linestyle="-.", linewidth=1.5)
        ax_main.plot(v_win, phi_np[i_line, i_point, mask], color=colors[i_point], linestyle="-", linewidth=2.0, label="SkewerJAX" if i_point == 0 else None)
        ax_main.plot(v_win, phi_analytic_np[i_line, i_point, mask], color=colors[i_point], linestyle=":", linewidth=4.0, alpha=0.6, label="Analytical" if i_point == 0 else None)

        residual = phi_np[i_line, i_point, mask] / phi_analytic_np[i_line, i_point, mask] - 1.0
        ax_res.plot(v_win, residual, color=colors[i_point], linestyle="-", linewidth=0.5, alpha=0.4)

    ax_res.axhline(0.0, color="k", linewidth=1, alpha=0.5); ax_res.fill_between(v_win, -1e-6, 1e-6, color="grey", alpha=0.2)
    ax_res.hlines(1e-6, xmin=v_win[0], xmax=v_win[-1], color="k", linewidth=1, linestyle="--", alpha=0.5); ax_res.hlines(-1e-6, xmin=v_win[0], xmax=v_win[-1], color="k", linewidth=1, linestyle="--", alpha=0.5)
    ax_res.set_xlabel(r"Line-centered velocity $v^{(j)}\ [\mathrm{km\,s^{-1}}]$"); ax_res.set_ylabel("residual"); ax_res.set_yscale("symlog", linthresh=1e-6); ax_res.set_ylim(-1e-2, 1e-2)
    ax_res.set_yticks([-1e-2, -1e-5, 0.0, 1e-5, 1e-2]); ax_res.yaxis.set_major_formatter(mpl.ticker.FuncFormatter(sjx.utils.plotting_tools.ten_power_formatter)); ax_res.yaxis.set_minor_formatter(mpl.ticker.NullFormatter())
    ax_main.set_title(rf"{line.name}, $\lambda_0 = {line.lambda0_A:.3f}\,\mathrm{{\AA}}$", fontsize=20); ax_main.set_yscale("log"); ax_main.set_ylabel(r"$\phi(v^{(j)})\; \left[\mathrm{ns}\right]$")

handles = [
    Line2D([0], [0], color=colors[i], lw=4, linestyle="-", label=rf"$a/a_\dagger = {scale_factor[i]:.5g}/{a_dagger_per_point[i]:.5g}$" + "\n" + rf"$v_\mathrm{{peak}}={c_kms*np.log(a_dagger_per_point[i]/scale_factor[i]):.2f}\,\mathrm{{km\,s^{{-1}}}}$")
    for i in range(N_points)
]
axes[0, 0].legend(handles=handles, loc="upper left", fontsize=12, ncols=1)

fig.tight_layout()
fig.subplots_adjust(hspace=0.05, wspace=0.2)
plt.show()

# ToDo

# Check variations with temperature and LoS velocity

### Define the fields to sample

In [ ]:
L_box = 1000.0 # UNITS: [Mpc]

fields = Fields({
    "HI": {
        "temperature": ScalarField(toy_samplers.temperature_sampler(T0_min=1e0, T0_max=1e6, L_box=L_box)), # UNITS: [K]
        "velocity": VectorField(toy_samplers.velocity_sampler(v0_min=-300.0, v0_max=300.0, L_box=L_box)),  # UNITS: [km/s]
        "turbulence": ScalarField(toy_samplers.turbulence_sampler(b_turb=0.0)),                            # UNITS: [km/s]
        "collisional": ScalarField(toy_samplers.collisional_sampler(A_coll=0.0))                           # UNITS: [GHz]
    },
    "SiIII": {
        "temperature": ScalarField(toy_samplers.temperature_sampler(T0_min=1e0, T0_max=1e6, L_box=L_box)), # UNITS: [K]
        "velocity": VectorField(toy_samplers.velocity_sampler(v0_min=-300.0, v0_max=300.0, L_box=L_box)),  # UNITS: [km/s]
        "turbulence": ScalarField(toy_samplers.turbulence_sampler(b_turb=0.0)),                            # UNITS: [km/s]
        "collisional": ScalarField(toy_samplers.collisional_sampler(A_coll=0.0))                           # UNITS: [GHz]
    }
})

### Define wavelength range and lines to study

In [ ]:
wavelengths = jnp.linspace(970, 1280, 2**18)  # in Angstroms
line_bundle = sjx.constants.lines.LineBundle(["HI-LyA", "HI-LyB", "SiIII-1206"])

### Define points to evaluate

In [ ]:
N_sk = 7
N_ppsk = 5
N_grid = 64

points = aux.make_plane_points(L_box, 'y', L_box/2, N_sk, N_ppsk)  # spatial points corresponding to N_sk skewers of length N_ppsk along the y-plane with y = L_box/2, shape (N_sk, N_ppsk, 3)
scale_factors = jnp.ones(points.shape[0:-1])[..., None]
quadri_vectors = jnp.concatenate([scale_factors, points], axis=-1) # quadri-vectors corresponding to N_sk skewers considering constant scale factors = 1.0, shape (N_sk, N_ppsk, 4)

LoS_directions = jnp.zeros_like(points)
LoS_directions = LoS_directions.at[..., 0].set(1.0)                # line-of-sight directions corresponding to N_sk skewers pointing towards the x-direction, shape (N_sk, N_ppsk, 3)

scale_factor_obs = jnp.ones(quadri_vectors.shape[0:-1])            # scale factor wrt the observer, here we are assuming no cosmological expansion, a0=1

### Compute line profiles

In [ ]:
phi_ns = line_profile.compute_line_profiles(
    quadri_vectors,
    LoS_directions,
    scale_factor_obs,
    wavelengths,
    line_bundle,
    fields,
    voigt_implementation_mode = "ExoJAX",
)
print("Shape of phi_ns", phi_ns.shape, ". Expected shape (N_lines, *batch_shape, N_lambda) = (",line_bundle.n_lines,",",quadri_vectors.shape[:-1],",",len(wavelengths),")")

### Plot showing temperature variations

In [ ]:
# Convert JAX arrays to NumPy for plotting
lambdas_np = np.asarray(wavelengths)           # (N_lambda,)
phi_np = np.asarray(phi_ns)                    # (N_sp, N_sk, N_ppsk, N_lambda)

colors = np.array(plt.cm.get_cmap("tab10").colors[:N_sk])
linestyles = ['-', '--', ':']

i_ppsk = N_ppsk // 2

fig, ax = plt.subplots(figsize=(14, 6))

# Plot all profiles on the same axes
for i_sp, line in enumerate(line_bundle.lines):
    ax.axvline(line.lambda0_A, color="grey", linestyle="-.", linewidth=1.5, alpha=1.0)
    ls = linestyles[i_sp % len(linestyles)]
    for i_sk in range(N_sk):
        profile = phi_np[i_sp, i_sk, i_ppsk, :]   # full wavelength range
        ax.plot(lambdas_np, profile, color=colors[i_sk], linestyle=ls, linewidth=2.0)

# Axes formatting
ax.set_yscale("log")
ax.set_xlabel(r"Wavelength $\lambda\ [\mathrm{\AA}]$")
ax.set_ylabel(r"$\phi(\lambda)\; \left[\mathrm{ns}\right]$")
ax.set_title("Line profiles for different transitions and temperatures", fontsize=20)

# Legend for temperatures (colors)
temp_handles = []
for i_sk in range(N_sk):
    _sampled_T = fields.temperature(quadri_vectors[i_sk])
    lbl = rf"$T = {_sampled_T[0, 0]:.2g}\,\mathrm{{K}}$"
    temp_handles.append(Line2D([0], [0], color=colors[i_sk], lw=4, linestyle="-", label=lbl))
leg_T = ax.legend(handles=temp_handles, loc="upper left", fontsize=16, ncols=1)
ax.add_artist(leg_T)


# Legend for lines (linestyles)
line_handles = []
for i_sp, line in enumerate(line_bundle.lines):
    ls = linestyles[i_sp % len(linestyles)]
    lbl = rf"{line.name},  $\lambda_0 = {line.lambda0_A:.3f}\,\mathrm{{\AA}}$"
    line_handles.append(Line2D([0], [0], color="k", lw=3, linestyle=ls, label=lbl))
leg_L = ax.legend(handles=line_handles, loc="lower right", fontsize=16, ncol=line_bundle.n_lines)

fig.tight_layout()
plt.show()



# Zoom around each line center [Angstrom]
delta_lambda_A = 1.3   # adjust to taste
fig, axes = plt.subplots(nrows=1, ncols=line_bundle.n_lines, figsize=(14, 6), sharex=False, sharey=True)
if line_bundle.n_lines == 1:
    axes = [axes]  # in case N_sp=1, make it iterable

for i_sp, (ax, line) in enumerate(zip(axes, line_bundle.lines)):
    lambda0 = line.lambda0_A
    ax.axvline(lambda0, color="grey", linestyle="-.", linewidth=1.5)

    mask = (lambdas_np >= lambda0 - delta_lambda_A) & (lambdas_np <= lambda0 + delta_lambda_A)
    lam_win = lambdas_np[mask]  # (N_lambda_window,)

    for i_sk in range(N_sk):
        profile = phi_np[i_sp, i_sk, i_ppsk, mask]  # (N_lambda_window,)
        ax.plot(lam_win, profile, color=colors[i_sk], linestyle=linestyles[i_sp], linewidth=2.0)

    # Panel title and scaling
    ax.set_title(rf"{line.name},  $\lambda_0 = {line.lambda0_A:.3f}\,\mathrm{{\AA}}$", fontsize=20)
    ax.set_yscale("log")
    ax.set_xlabel(r"Wavelength $\lambda\ [\mathrm{\AA}]$")
axes[0].set_ylabel(r"$\phi(\lambda)\; \left[\mathrm{ns}\right]$")

fig.tight_layout()
plt.show()

### Plot showing $v_{LoS}$ variations

In [ ]:
# Convert JAX arrays to NumPy for plotting
lambdas_np = np.asarray(wavelengths)                          # (N_lambda,)
phi_np = np.asarray(phi_ns)                                    # (N_sp, N_sk, N_ppsk, N_lambda)

colors = np.array(plt.cm.get_cmap("tab10").colors[:N_sk])
linestyles = ['-', '--', ':']

i_sk = N_sk // 2

fig, ax = plt.subplots(figsize=(14, 6))

# Plot all profiles on the same axes
for i_sp, line in enumerate(line_bundle.lines):
    ax.axvline(line.lambda0_A, color="grey", linestyle="-.", linewidth=1.5, alpha=1.0)
    ls = linestyles[i_sp % len(linestyles)]
    for i_ppsk in range(N_ppsk):
        profile = phi_np[i_sp, i_sk, i_ppsk, :]   # full wavelength range
        ax.plot(lambdas_np, profile, color=colors[i_ppsk], linestyle=ls, linewidth=2.0)

# Axes formatting
ax.set_yscale("log")
ax.set_xlabel(r"Wavelength $\lambda\ [\mathrm{\AA}]$")
ax.set_ylabel(r"$\phi(\lambda)\; \left[\mathrm{ns}\right]$")
ax.set_title("Line profiles for different transitions and LoS velocities", fontsize=20)

# Legend for velocities (colors)
handles = []
for i_ppsk in range(N_ppsk):
    _sampled_fields = fields.sample(quadri_vectors[:, i_ppsk], LoS_directions[:, i_ppsk])
    _sampled_vLoS = _sampled_fields['HI']["LoS_velocity"][0]
    lbl = r"$v_\mathrm{{LOS}} = $" + str(round(_sampled_vLoS, 1)) + r" $\mathrm{km/s}$"
    handles.append(Line2D([0], [0], color=colors[i_ppsk], lw=4, linestyle="-", label=lbl))
leg = ax.legend(handles=handles, loc="upper left", fontsize=16, ncols=1)
ax.add_artist(leg)


# Legend for lines (linestyles)
line_handles = []
for i_sp, line in enumerate(line_bundle.lines):
    ls = linestyles[i_sp % len(linestyles)]
    lbl = rf"{line.name},  $\lambda_0 = {line.lambda0_A:.3f}\,\mathrm{{\AA}}$"
    line_handles.append(Line2D([0], [0], color="k", lw=3, linestyle=ls, label=lbl))
leg_L = ax.legend(handles=line_handles, loc="lower right", fontsize=16, ncol=line_bundle.n_lines)

fig.tight_layout()
plt.show()


# Zoom around each line center [Angstrom]
delta_lambda_A = 1.3   # adjust to taste
fig, axes = plt.subplots(nrows=1, ncols=line_bundle.n_lines, figsize=(14, 6), sharex=False, sharey=True)
if line_bundle.n_lines == 1:
    axes = [axes]  # in case N_sp=1, make it iterable

for i_sp, (ax, line) in enumerate(zip(axes, line_bundle.lines)):
    lambda0 = line.lambda0_A
    ax.axvline(lambda0, color="grey", linestyle="-.", linewidth=1.5)

    mask = (lambdas_np >= lambda0 - delta_lambda_A) & (lambdas_np <= lambda0 + delta_lambda_A)
    lam_win = lambdas_np[mask]  # (N_lambda_window,)

    for i_ppsk in range(N_ppsk):
        profile = phi_np[i_sp, i_sk, i_ppsk, mask]  # (N_lambda_window,)
        ax.plot(lam_win, profile, color=colors[i_ppsk], linestyle=linestyles[i_sp], linewidth=2.0)

    # Panel title and scaling
    ax.set_title(rf"{line.name},  $\lambda_0 = {line.lambda0_A:.3f}\,\mathrm{{\AA}}$", fontsize=20)
    ax.set_yscale("log")
    ax.set_xlabel(r"Wavelength $\lambda\ [\mathrm{\AA}]$")
axes[0].set_ylabel(r"$\phi(\lambda)\; \left[\mathrm{ns}\right]$")

fig.tight_layout()
plt.show()